[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/photoelasticity/photoelasticity_poincare.ipynb)

# Photoelasticity on the Poincaré Sphere

This example computes the photoelastic fringe patterns of a loaded transparent plate with a circular hole by tracking polarization states on the Poincaré sphere. Pulling the plate along its length concentrates stress around the rim of the hole and makes the material birefringent, so light polarized along one principal stress direction lags behind light polarized along the other and produces dark and bright fringes between polarizers.

In-plane stress is modelled as a linear map from extrinsic cut planes to intrinsic traction directions built from metric-free dyads `radial * (Cut & radial)`, while solving `(Cut & stress(Cut)).eigh(Cut | Cut)` against the spatial metric gives the principal stresses and principal cuts. Polarization states are modelled as unit vectors on the Poincaré sphere in the same three-dimensional algebra as the plate, where linear polarizations sit on the equator at twice their angle in the plate and circular polarizations sit at the poles. Birefringent retardance acts as a spatial rotor `retarder >> state` turning the sphere by the phase difference between the principal stresses, and an analyser transmits `(1 + (analyser | state)) / 2`. The companion [spacetime photoelasticity notebook](photoelasticity.ipynb) computes the same fringes directly on the oscillating electric field bivector in spacetime.

In Cauchy tensor and Mueller-calculus notation, transforming Kirsch's polar stresses into Cartesian components gives the principal angle $\alpha = \tfrac12\operatorname{atan2}(2\sigma_{xy}, \sigma_{xx} - \sigma_{yy})$ and retardance $\delta = 2\pi C t(\sigma_1 - \sigma_2)/\lambda$, and propagating the four-component Stokes vector $S = (S_0, S_1, S_2, S_3)^{\mathsf T}$ multiplies $4 \times 4$ rotation and retarder Mueller matrices $M(-2\alpha)\,M_{\mathrm{ret}}(\delta)\,M(2\alpha)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, concatenate
from numga.algebras import VGA3D
from examples.util.animation import save_figure, show_animation
from examples.optics.photoelasticity import render

ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                  # Scalar+
Cut = ga.gatype.vector().derive.within(Algebra("x+y+"))      # Vector+
Planar = Cut.derive.dual                                     # Antivector-
Traction = Planar                                            # Antivector-
Stress = ga.gatype((Traction, Cut))                          # Traction- <- Cut+
Polarization = ga.gatype.vector()                            # Vector+
Bivector = ga.gatype.bivector()                              # Bivector+

eighth_turn = (mv.xy * (-np.pi / 8)).exp()                   # [] Rotor+
quarter_turn = (mv.xy * (-np.pi / 4)).exp()                  # [] Rotor+

## Stress concentration around a circular hole

Pulling the plate along `x.dual()` far from the circular hole redistributes traction around the unloaded rim so that both radial and shear tractions vanish on the boundary. Summing the radial, tangential, and shear dyads forms Kirsch's stress map without invoking a metric, and solving `(Cut & stress(Cut)).eigh(Cut | Cut)` against the spatial metric yields the principal stresses and principal directions.

In [ ]:
radius = 10.0
extent = 30.0
n_pixels = 200
tension = 30.0

In [ ]:
def kirsch_stress(positions: Planar, radius: float, remote: Stress) -> Stress:
    """Kirsch's exterior solution maps `positions` (`[...] Planar-`) outside a traction-free hole of `radius` under
    `remote` load to the local stress (`[...] Traction- <- Cut+`)."""
    distance = positions.antinorm()                                               # [...] Scalar+
    radial = positions / distance                                                 # [...] Planar-
    tangent = quarter_turn >> radial                                              # [...] Planar-
    radius_ratio = (radius / distance).squared()                                  # [...] Scalar+

    radial_cut, tangent_cut = radial.dual_inverse(), tangent.dual_inverse()       # [...] Cut+, Cut+
    mean = (Cut & remote(Cut)).contract() / 2                                     # [...] Scalar+
    anisotropy = (radial_cut & remote(radial_cut)) - mean                         # [...] Scalar+
    shear = radial_cut & remote(tangent_cut)                                      # [...] Scalar+
    radial_stress = (mean * (1 - radius_ratio)
                     + anisotropy * (1 - 4 * radius_ratio + 3 * radius_ratio.squared()))  # [...] Scalar+
    tangential_stress = (mean * (1 + radius_ratio)
                         - anisotropy * (1 + 3 * radius_ratio.squared()))         # [...] Scalar+
    shear_stress = shear * (1 + 2 * radius_ratio - 3 * radius_ratio.squared())    # [...] Scalar+
    return (radial_stress * radial * (Cut & radial)
            + tangential_stress * tangent * (Cut & tangent)
            + shear_stress * (radial * (Cut & tangent) + tangent * (Cut & radial)))  # [...] Traction- <- Cut+


def principal_stresses(stress: Stress) -> tuple[Scalar, Planar]:
    """Solving `(Cut & stress(Cut)).eigh(Cut | Cut)` yields the ascending `[..., 2] Scalar+` principal stresses and
    `[..., 2] Planar-` principal directions."""
    principal, cuts = (Cut & stress(Cut)).eigh(Cut | Cut)                         # [..., 2] Scalar+, Cut+
    return principal, cuts.dual()                                                 # [..., 2] Scalar+, Planar-


centres = (np.arange(n_pixels) + 0.5) * (2 * extent / n_pixels) - extent
positions = mv.x.dual() * centres[None, :] + mv.y.dual() * centres[:, None]       # [n_pixels, n_pixels] Planar-
pull_direction = mv.x.dual()                                                      # [] Planar-
remote = tension * pull_direction * (Cut & pull_direction)                        # [] Traction- <- Cut+
stress = kirsch_stress(positions, radius, remote)                                 # [n_pixels, n_pixels] Traction- <- Cut+
principal, directions = principal_stresses(stress)                                # [n_pixels, n_pixels, 2] Scalar+, Planar-
difference = principal[..., 1] - principal[..., 0]                                # [n_pixels, n_pixels] Scalar+
save_figure(render.stress_figure(positions, difference, radius, tension), "photoelastic_poincare_stress");

In the figure, colour shows the principal stress difference in units of the applied tension across a square window spanning three hole radii from the centre of the infinite plate, rising from zero in dark blue to three in yellow at the top and bottom of the rim. [MIT's hole-in-a-plate experiment](https://ocw.mit.edu/courses/2-002-mechanics-and-materials-ii-spring-2004/09e8137971e754b1b7290a33a100bb05_lab_4_s04.pdf) gives the analytical Kirsch derivation and experimental setup.

## Retardance on the Poincaré sphere

On the Poincaré sphere, linear polarizations lie around the equator in the plate's plane at twice their physical angle so opposite directions map to the same state, while the two circular polarizations sit at the poles along `z`. Birefringence rotates the sphere about the equatorial state of the principal stress direction by the phase retardance proportional to `difference`, and an analyser passes `(1 + (analyser | state)) / 2` by Malus's law. Crossed linear polarizers go dark wherever the incident state lies on the rotation axis, whereas circular light entering at `z` sits at equal angle from every equatorial axis and isolates the stress-difference fringes.

In Jones-calculus notation, removing the mean phase leaves the $\mathrm{SU}(2)$ retarder $\operatorname{diag}(e^{i\delta/2}, e^{-i\delta/2})$ in the principal frame, whose double cover acts as an $\mathrm{SO}(3)$ rotation on the Stokes vector.

In [ ]:
thickness = 1.0
wavelength = 550e-6
stress_optic = 5.5e-5
polarizer_angles = np.deg2rad([0, 22.5])
delay_per_stress = 2 * np.pi * stress_optic * thickness / wavelength

In [ ]:
def polarization(direction: Planar) -> Polarization:
    """Doubling the angle of unit `direction` (`[...] Planar-`) from `x.dual()` places its linear state on the
    Poincaré equator (`[...] Polarization+`)."""
    return (direction * mv.x.dual().inverse()) >> mv.x                            # [...] Polarization+


def transmitted_intensity(state: Polarization, analyser: Polarization) -> Scalar:
    """Malus's law `(1 + (analyser | state)) / 2` gives the transmitted intensity (`[...] Scalar+`) of `state`
    through `analyser`."""
    return (1 + (analyser | state)) / 2                                           # [...] Scalar+


retardance = delay_per_stress * difference                                        # [n_pixels, n_pixels] Scalar+
# Plane orthogonal to the second principal state, spanned by the state an eighth turn away and the circular pole.
half_turn = (polarization(eighth_turn >> directions[..., 1]) ^ mv.z) * (retardance / 2)   # [n_pixels, n_pixels] Bivector+
plate_rotors = half_turn.exp()                                                    # [n_pixels, n_pixels] Rotor+

polarizer_rotors = (mv.xy * (-polarizer_angles / 2)).exp()                        # [2] Rotor+
linear_states = polarization(polarizer_rotors >> mv.x.dual())                     # [2] Polarization+
incident_states = concatenate((linear_states, mv.z[None]))                        # [3] Polarization+
analyser_states = -incident_states                                                # [3] Polarization+
exit_states = plate_rotors >> incident_states[:, None, None]                      # [3, n_pixels, n_pixels] Polarization+
intensities = transmitted_intensity(exit_states, analyser_states[:, None, None])  # [3, n_pixels, n_pixels] Scalar+
save_figure(render.polariscope_figure(positions, intensities, radius), "photoelastic_poincare_polariscopes");

From left to right, the three panels show the crossed linear polariscope aligned with the load, the crossed linear polariscope turned by 22.5 degrees, and the dark-field circular polariscope, matching the spacetime electric-field views.

## Loading the plate

Ramping the applied tension from zero to its peak value and back keeps each pixel's rotation axis on the Poincaré equator fixed while scaling its turning angle, so fringes emerge from the rim during loading and retreat during unloading.

In [ ]:
n_frames = 180
n_load_steps = n_frames // 2 + 1
duration_ms = 40
n_colors = 12
loads = (1 - np.cos(np.linspace(0, np.pi, n_load_steps))) / 2                    # [n_load_steps]

In [ ]:
def loading_frames(half_turn: Bivector, loads: np.ndarray) -> Iterator[Scalar]:
    """Yields `[n_pixels, n_pixels] Scalar+` circular-polariscope intensities across `loads`."""
    for load in loads:
        loaded_rotors = (half_turn * load).exp()                                  # [n_pixels, n_pixels] Rotor+
        yield transmitted_intensity(loaded_rotors >> mv.z, -mv.z)                 # [n_pixels, n_pixels] Scalar+


show_animation(render.polariscope_animation(positions, loading_frames(half_turn, loads), radius),
               "photoelastic_poincare_loading", duration_ms, colors=n_colors, forward_then_back=True);

In [ ]:
# checks
outside = positions.antinorm() >= radius
symmetry_residual = (mv.x & stress(mv.y)) - (mv.y & stress(mv.x))                 # [n_pixels, n_pixels] Scalar+
assert (symmetry_residual[outside].abs() < 1e-10).all()
assert ((exit_states[:, outside].norm() - 1).abs() < 1e-10).all()

expected_circular = (retardance / 2).sin().squared()                              # [n_pixels, n_pixels] Scalar+
assert ((intensities[2][outside] - expected_circular[outside]).abs() < 1e-10).all()
assert (intensities[:, outside] >= -1e-10).all() and (intensities[:, outside] <= 1 + 1e-10).all()
assert (intensities[:2, outside] <= intensities[2][outside] + 1e-10).all()